# EduYou — Get Embeddings (Azure OpenAI)

**File:** `get_embeddings_eduyou.ipynb`  
**Generated:** 2026-03-30

This notebook is a drop-in adaptation of the professor’s *get_embeddings* template for the EduYou capstone. It creates vector embeddings for **one row per document** from:

- `cleaned/eduyou_cip_docs_for_embedding.csv`

and writes an embeddings table compatible with the professor’s `07_RAG_query.ipynb` pattern (doc_id + `dim_*` columns).

---

## Column-by-column mapping (EduYou → RAG template)

| EduYou CSV column | Role in RAG template | Used how |
|---|---|---|
| `text` | document text | embedded into a vector |
| `doc_id` | document identifier | stored alongside embedding |
| `cip4`, `degree_level`, `cip_title`, `median_earnings_4yr_nat` | metadata | carried into output for filtering/debugging |

> ✅ **Important:** Do *not* embed the raw multi-million-row join tables. Use the document-level file (`eduyou_cip_docs_for_embedding.csv`).

---

## Embedding model configuration (Professor note)

Set `AZURE_OPENAI_DEPLOYMENT_ID` to one of:
- `text-embedding-ada-002`
- `text-embedding-3-small`
- `text-embedding-3-large`

Deployments are created using the same names as the models, so you can use those names directly.


In [ ]:
# If running in Colab
# from google.colab import drive
# drive.mount('/content/drive')

import os
print('Working dir:', os.getcwd())
print('Files:', os.listdir())


## 1) Install dependencies


In [ ]:
!pip -q install openai pandas numpy

## 2) Configuration

Set your Azure OpenAI endpoint and key securely.
- Recommended: store in environment variables.
- Do **not** hardcode secrets in notebooks.


In [ ]:
import os
import getpass
import pandas as pd
import numpy as np
from openai import AzureOpenAI

# --- Required settings ---
AZURE_OPENAI_ENDPOINT = os.getenv('AZURE_OPENAI_ENDPOINT') or getpass.getpass('AZURE_OPENAI_ENDPOINT (e.g., https://<resource>.openai.azure.com/): ')
AZURE_OPENAI_API_KEY  = os.getenv('AZURE_OPENAI_API_KEY')  or getpass.getpass('AZURE_OPENAI_API_KEY: ')
AZURE_OPENAI_API_VERSION = os.getenv('AZURE_OPENAI_API_VERSION') or '2024-02-01'

# --- Deployment/model name (per professor note) ---
AZURE_OPENAI_DEPLOYMENT_ID = os.getenv('AZURE_OPENAI_DEPLOYMENT_ID') or 'text-embedding-3-small'

# Optional: for text-embedding-3-* you may set dimensions to shorten vectors.
# Leave as None to use model default (3-small: 1536, 3-large: 3072).
EMBEDDING_DIMENSIONS = None

# Input/output paths
INPUT_DOCS_CSV = 'cleaned/eduyou_cip_docs_for_embedding.csv'
OUTPUT_DIR = 'embeddings'
os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_EMB_CSV = os.path.join(OUTPUT_DIR, f"eduyou_embeddings_{AZURE_OPENAI_DEPLOYMENT_ID}.csv")

print('Model/deployment:', AZURE_OPENAI_DEPLOYMENT_ID)
print('Input docs:', INPUT_DOCS_CSV)
print('Output embeddings:', OUTPUT_EMB_CSV)


## 3) Load EduYou document table

This file must have at minimum:
- `doc_id`
- `text`


In [ ]:
docs = pd.read_csv(INPUT_DOCS_CSV, low_memory=False)

required_cols = {'doc_id','text'}
missing = required_cols - set(docs.columns)
if missing:
    raise ValueError(f"Missing required columns in {INPUT_DOCS_CSV}: {missing}")

print('Rows (documents):', len(docs))
print('Columns:', docs.columns.tolist())

docs.head()


## 4) Create embeddings (batched)

This follows the professor’s Azure OpenAI embedding example using `AzureOpenAI`.
We batch requests to reduce overhead.

The output file uses:
- `doc_id`
- metadata columns (if present)
- embedding columns named `dim_0 ... dim_{p-1}`

The notebook can resume if partially completed.


In [ ]:
client = AzureOpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_OPENAI_API_VERSION,
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
)

# Metadata columns to carry through if present
meta_cols = [c for c in ['cip4','degree_level','cip_title','median_earnings_4yr_nat'] if c in docs.columns]

BATCH_SIZE = 64

# Resume support
if os.path.exists(OUTPUT_EMB_CSV):
    emb_df = pd.read_csv(OUTPUT_EMB_CSV)
    done_ids = set(emb_df['doc_id'].astype(str)) if 'doc_id' in emb_df.columns else set()
    start_idx = docs['doc_id'].astype(str).isin(done_ids).sum()
    print(f"Found existing embeddings file with {len(done_ids)} doc_id rows. Will skip embedded docs.")
else:
    emb_df = None
    done_ids = set()
    start_idx = 0

# Helper to call embeddings API
def get_embeddings_batch(text_list):
    kwargs = {
        'model': AZURE_OPENAI_DEPLOYMENT_ID,
        'input': text_list,
    }
    # dimensions parameter is supported for text-embedding-3-* models
    if EMBEDDING_DIMENSIONS is not None:
        kwargs['dimensions'] = int(EMBEDDING_DIMENSIONS)

    resp = client.embeddings.create(**kwargs)
    return [d.embedding for d in resp.data]

# Determine embedding dimension from first batch
first_text = docs.loc[0, 'text']
first_vec = get_embeddings_batch([first_text])[0]
P = len(first_vec)
print('Embedding dimension P =', P)

dim_cols = [f'dim_{i}' for i in range(P)]

# If starting fresh, initialize file with headers
if not os.path.exists(OUTPUT_EMB_CSV):
    cols = ['doc_id'] + meta_cols + dim_cols
    pd.DataFrame(columns=cols).to_csv(OUTPUT_EMB_CSV, index=False)

# Process documents
for i in range(len(docs)):
    doc_id = str(docs.loc[i, 'doc_id'])
    if doc_id in done_ids:
        continue

    # Batch indices
    batch_start = i
    batch_end = min(i + BATCH_SIZE, len(docs))

    # Build batch filtering out already-done ids
    batch_rows = docs.iloc[batch_start:batch_end].copy()
    batch_rows['doc_id'] = batch_rows['doc_id'].astype(str)
    batch_rows = batch_rows[~batch_rows['doc_id'].isin(done_ids)]

    if batch_rows.empty:
        continue

    texts = batch_rows['text'].astype(str).tolist()
    vecs = get_embeddings_batch(texts)

    out_rows = []
    for (idx, row), vec in zip(batch_rows.iterrows(), vecs):
        rec = {'doc_id': str(row['doc_id'])}
        for mc in meta_cols:
            rec[mc] = row.get(mc, '')
        rec.update({dim_cols[j]: float(vec[j]) for j in range(P)})
        out_rows.append(rec)
        done_ids.add(str(row['doc_id']))

    pd.DataFrame(out_rows).to_csv(OUTPUT_EMB_CSV, mode='a', header=False, index=False)

    # advance loop by batch size
    i = batch_end - 1

print('Done. Saved embeddings to:', OUTPUT_EMB_CSV)


## 5) Quick sanity check

Load the embeddings output and confirm shape.


In [ ]:
emb = pd.read_csv(OUTPUT_EMB_CSV)
print('Embedding table shape:', emb.shape)
print('Head:')
emb.head()


## Next step

Use the resulting embeddings CSV in the professor’s `07_RAG_query.ipynb` workflow:
- Load this embeddings file
- Load the same document file (`eduyou_cip_docs_for_embedding.csv`)
- Compute query embedding with the same `AZURE_OPENAI_DEPLOYMENT_ID`
- Retrieve Top‑K by distance/similarity
